In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, precision_recall_curve,
                             average_precision_score)
from imblearn.over_sampling import SMOTE
import river
from river import anomaly, preprocessing, stream
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Generate Realistic Transaction Dataset
np.random.seed(42)
n_legit = 9700
n_fraud = 300

# Legitimate transactions
legit = pd.DataFrame({
    'amount':        np.random.exponential(80, n_legit),
    'hour':          np.random.choice(range(8, 22), n_legit),
    'day_of_week':   np.random.choice(range(0, 5), n_legit),
    'merchant_risk': np.random.uniform(0.0, 0.3, n_legit),
    'distance_km':   np.random.exponential(10, n_legit),
    'prev_tx_mins':  np.random.exponential(120, n_legit),
    'is_foreign':    np.random.choice([0,1], n_legit, p=[0.95, 0.05]),
    'label': 0
})

# Fraudulent transactions — distinct behavioral patterns
fraud = pd.DataFrame({
    'amount':        np.random.exponential(400, n_fraud),
    'hour':          np.random.choice(range(0, 6), n_fraud),
    'day_of_week':   np.random.choice(range(5, 7), n_fraud),
    'merchant_risk': np.random.uniform(0.6, 1.0, n_fraud),
    'distance_km':   np.random.exponential(800, n_fraud),
    'prev_tx_mins':  np.random.exponential(5, n_fraud),
    'is_foreign':    np.random.choice([0,1], n_fraud, p=[0.3, 0.7]),
    'label': 1
})

df = pd.concat([legit, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset shape:    {df.shape}")
print(f"Fraud rate:       {df['label'].mean()*100:.1f}%")
print(f"\nLegit transactions:  {n_legit:,}")
print(f"Fraud transactions:  {n_fraud:,}")
print(f"\nFeature stats:")
df.groupby('label')[['amount','hour','distance_km','prev_tx_mins']].mean().round(2)

Dataset shape:    (10000, 8)
Fraud rate:       3.0%

Legit transactions:  9,700
Fraud transactions:  300

Feature stats:


,amount,hour,distance_km,prev_tx_mins
label,,,,
0,78.12,14.44,9.96,119.35
1,384.27,2.46,759.45,5.16


In [3]:
# Cell 3 — Feature Engineering & SMOTE Resampling
# Engineer additional features
df['amount_log']       = np.log1p(df['amount'])
df['high_risk_hour']   = (df['hour'].between(0, 6)).astype(int)
df['rapid_succession'] = (df['prev_tx_mins'] < 10).astype(int)
df['high_distance']    = (df['distance_km'] > 500).astype(int)
df['risk_score_raw']   = (df['merchant_risk'] * 
                          df['is_foreign'] * 
                          df['high_risk_hour'])

features = ['amount_log', 'hour', 'day_of_week', 'merchant_risk',
            'distance_km', 'prev_tx_mins', 'is_foreign',
            'high_risk_hour', 'rapid_succession', 
            'high_distance', 'risk_score_raw']

X = df[features]
y = df['label']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE — oversample minority class for training only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print(f"Original train shape:  {X_train_scaled.shape}")
print(f"After SMOTE:           {X_train_sm.shape}")
print(f"\nClass balance after SMOTE:")
print(f"  Legit: {(y_train_sm==0).sum():,}")
print(f"  Fraud: {(y_train_sm==1).sum():,}")
print(f"\nTest set (untouched):")
print(f"  Legit: {(y_test==0).sum():,}")
print(f"  Fraud: {(y_test==1).sum():,}")

Original train shape:  (8000, 11)
After SMOTE:           (15520, 11)

Class balance after SMOTE:
  Legit: 7,760
  Fraud: 7,760

Test set (untouched):
  Legit: 1,940
  Fraud: 60


In [4]:
# Cell 4 — Train Models: Isolation Forest + Random Forest + Logistic Regression
# Isolation Forest (unsupervised anomaly detection)
iso = IsolationForest(contamination=0.03, random_state=42, n_estimators=200)
iso.fit(X_train_scaled)
iso_scores = -iso.score_samples(X_test_scaled)

# Random Forest (supervised)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=8,
    class_weight='balanced', random_state=42
)
rf.fit(X_train_sm, y_train_sm)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Logistic Regression (baseline)
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_sm, y_train_sm)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

# Normalize Isolation Forest scores to [0,1]
iso_proba = (iso_scores - iso_scores.min()) / (iso_scores.max() - iso_scores.min())

# Evaluate
models = {
    'Isolation Forest': iso_proba,
    'Random Forest':    rf_proba,
    'Logistic Regression': lr_proba
}

print(f"{'Model':<25} {'ROC-AUC':>10} {'Avg Precision':>15}")
print("-" * 52)
for name, proba in models.items():
    auc = roc_auc_score(y_test, proba)
    ap  = average_precision_score(y_test, proba)
    print(f"{name:<25} {auc:>10.4f} {ap:>15.4f}")

Model                        ROC-AUC   Avg Precision
----------------------------------------------------
Isolation Forest              1.0000          1.0000
Random Forest                 1.0000          1.0000
Logistic Regression           1.0000          1.0000


In [5]:
# Cell 5 — Precision-Recall Curve & Threshold Analysis
fig = go.Figure()

colors = {'Isolation Forest': 'cyan', 
          'Random Forest': 'lime',
          'Logistic Regression': 'orange'}

for name, proba in models.items():
    precision, recall, thresholds = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    fig.add_trace(go.Scatter(
        x=recall, y=precision,
        mode='lines',
        name=f'{name} (AP={ap:.4f})',
        line=dict(color=colors[name], width=2.5)
    ))

fig.update_layout(
    title='Precision-Recall Curve — Fraud Detection Models',
    xaxis_title='Recall (% of fraud caught)',
    yaxis_title='Precision (% of alerts that are real fraud)',
    template='plotly_dark',
    width=800, height=500
)
fig.show()

# Threshold analysis for Random Forest
print("\n=== Random Forest Threshold Analysis ===")
print(f"{'Threshold':>12} {'Precision':>12} {'Recall':>10} {'Alerts/day':>12}")
print("-" * 50)

precision, recall, thresholds = precision_recall_curve(y_test, rf_proba)
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    preds = (rf_proba >= threshold).astype(int)
    tp = ((preds==1) & (y_test==1)).sum()
    fp = ((preds==1) & (y_test==0)).sum()
    fn = ((preds==0) & (y_test==1)).sum()
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0
    rec  = tp/(tp+fn) if (tp+fn) > 0 else 0
    alerts = preds.sum()
    print(f"{threshold:>12.1f} {prec:>12.4f} {rec:>10.4f} {alerts:>12}")


=== Random Forest Threshold Analysis ===
   Threshold    Precision     Recall   Alerts/day
--------------------------------------------------
         0.3       1.0000     1.0000           60
         0.4       1.0000     1.0000           60
         0.5       1.0000     1.0000           60
         0.6       1.0000     1.0000           60
         0.7       1.0000     1.0000           60
         0.8       1.0000     1.0000           60


In [6]:
# Cell 6 — Simulate Real-Time Streaming Detection & Export
import os
from datetime import datetime, timedelta

# Simulate streaming transactions
print("=== Real-Time Fraud Detection Simulation ===\n")
stream_size  = 100
start_time   = datetime(2025, 4, 12, 9, 0, 0)
threshold    = 0.55
alerts       = []

stream_df = X_test.iloc[:stream_size].copy().reset_index(drop=True)
stream_scaled = X_test_scaled[:stream_size]
stream_labels = y_test.iloc[:stream_size].reset_index(drop=True)
stream_proba  = rf.predict_proba(stream_scaled)[:, 1]

for i in range(stream_size):
    tx_time  = start_time + timedelta(minutes=i*3)
    score    = stream_proba[i]
    is_fraud = stream_labels[i]
    flagged  = score >= threshold

    if flagged:
        alerts.append({
            'timestamp':   tx_time.strftime('%H:%M:%S'),
            'tx_index':    i,
            'risk_score':  round(score, 4),
            'actual_fraud':bool(is_fraud),
            'amount':      round(float(df['amount'].iloc[i]), 2),
            'alert_type':  'TRUE POSITIVE' if is_fraud else 'FALSE POSITIVE'
        })
        print(f"[{tx_time.strftime('%H:%M:%S')}] ALERT tx#{i:03d} "
              f"score={score:.4f} amount=${df['amount'].iloc[i]:.2f} "
              f"→ {alerts[-1]['alert_type']}")

print(f"\nStream processed: {stream_size} transactions")
print(f"Alerts fired:     {len(alerts)}")
tp = sum(1 for a in alerts if a['actual_fraud'])
fp = sum(1 for a in alerts if not a['actual_fraud'])
print(f"True positives:   {tp}")
print(f"False positives:  {fp}")

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-02-fraud-detection\outputs'
os.makedirs(output_dir, exist_ok=True)

pd.DataFrame(alerts).to_csv(f'{output_dir}\\stream_alerts.csv', index=False)
df[features + ['label']].to_csv(f'{output_dir}\\fraud_dataset.csv', index=False)

pd.DataFrame({
    'model': list(models.keys()),
    'roc_auc': [roc_auc_score(y_test, p) for p in models.values()],
    'avg_precision': [average_precision_score(y_test, p) for p in models.values()]
}).to_csv(f'{output_dir}\\model_comparison.csv', index=False)

print(f"\nExports saved to outputs/")

=== Real-Time Fraud Detection Simulation ===

[09:03:00] ALERT tx#001 score=1.0000 amount=$119.51 → TRUE POSITIVE
[10:51:00] ALERT tx#037 score=1.0000 amount=$38.90 → TRUE POSITIVE
[11:48:00] ALERT tx#056 score=1.0000 amount=$58.50 → TRUE POSITIVE
[13:51:00] ALERT tx#097 score=1.0000 amount=$155.44 → TRUE POSITIVE

Stream processed: 100 transactions
Alerts fired:     4
True positives:   4
False positives:  0

Exports saved to outputs/
